# גילוי תפקידי שחקנים חדשים ביורוליג באמצעות למידה לא־מונחית

## שאלת המחקר
האם ניתן לזהות מתוך נתוני הביצועים של שחקני יורוליג **תפקידי משחק טבעיים**, שאינם כפופים לחלוקה המסורתית ל־Guard, Forward ו־Center?

מטרת המחברת היא ליצור נקודת מבט חדשה על מבנה הקבוצה ותפקידי השחקנים, באמצעות אשכולות המבוססים על דפוסי משחק בפועל. עמודת העמדה המסורתית **אינה נכנסת לאימון**; היא משמשת רק לאחר יצירת האשכולות לצורך השוואה ופרשנות.

### מהלך העבודה
1. טעינה ובדיקת איכות הנתונים.
2. איחוד מדויק לפי `player_id`.
3. יצירת משתנים המייצגים סגנון משחק ל־36 דקות.
4. סינון עונות עם מדגם משחק קטן.
5. תקנון והפחתת ממדים באמצעות PCA.
6. בחירת מספר אשכולות בעזרת Silhouette Score ובדיקות נוספות.
7. יצירת אשכולות באמצעות K-Means והשוואה ל־Gaussian Mixture.
8. אפיון האשכולות והשוואתם לעמדות המסורתיות.
9. הכנה לצירוף נתוני 365Scores בהמשך.

> יחידת התצפית היא **שחקן–עונה–קבוצה**. שחקן שעבר קבוצה או שיחק במספר עונות עשוי להופיע יותר מפעם אחת, משום שתפקידו עשוי להשתנות.

## 1. התקנה וייבוא ספריות

In [ ]:
!pip -q install rapidfuzz openpyxl

import io, re, json, warnings, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (silhouette_score, calinski_harabasz_score,
                             davies_bouldin_score, adjusted_rand_score,
                             normalized_mutual_info_score)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
sns.set_theme(style='whitegrid', context='notebook')
RANDOM_STATE = 42

## 2. העלאת הקבצים

ב־Colab יש להעלות את שני הקבצים:
- `euroleague_players(1).csv` – סטטיסטיקה ברמת שחקן–עונה–קבוצה.
- `players_bio(1).csv` – גובה, עמדה מסורתית ופרטים ביוגרפיים.

אם הקבצים נמצאים ב־Google Drive, אפשר להחליף את ההעלאה בנתיבים מתוך Drive.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    print('Uploaded:', list(uploaded))
except ImportError:
    print('Not running in Colab; using files from the local upload folder if available.')

In [ ]:
def locate_file(filename):
    candidates = [Path(filename), Path('/content') / filename,
                  Path('upload') / filename]
    for p in candidates:
        if p.exists():
            return p
    matches = list(Path('.').rglob(filename))
    if matches:
        return matches[0]
    raise FileNotFoundError(f'לא נמצא הקובץ {filename}')

stats_path = locate_file('euroleague_players(1).csv')
bio_path = locate_file('players_bio(1).csv')

stats = pd.read_csv(stats_path)
bio = pd.read_csv(bio_path)
print('Stats:', stats.shape, '| Bio:', bio.shape)

## 3. בדיקת מבנה ואיכות

In [ ]:
def audit(df, name):
    return pd.DataFrame({
        'dataset': [name], 'rows': [len(df)], 'columns': [df.shape[1]],
        'duplicate_rows': [df.duplicated().sum()],
        'missing_cells': [int(df.isna().sum().sum())]
    })

display(pd.concat([audit(stats, 'statistics'), audit(bio, 'biography')], ignore_index=True))
display(stats.head(3))
display(bio.head(3))
display(bio.isna().sum().sort_values(ascending=False).head(10).to_frame('missing'))

## 4. איחוד לפי מזהה שחקן

שני המקורות מכילים `player_id`, ולכן אין צורך לבצע התאמה לפי שם. לפני החיבור נוודא שבקובץ הביוגרפי יש רשומה אחת בלבד לכל מזהה.

In [ ]:
bio_ok = bio[bio['fetch_status'].eq('ok')].copy()
bio_dupes = bio_ok[bio_ok.duplicated('player_id', keep=False)].sort_values('player_id')
print('Duplicate player IDs in valid bio rows:', bio_dupes['player_id'].nunique())

bio_one = (bio_ok.sort_values(['player_id', 'height_cm'], na_position='last')
           .drop_duplicates('player_id', keep='first'))

df = stats.merge(
    bio_one[['player_id','height_cm','position','nationality','born','first_name','last_name']],
    on='player_id', how='left', validate='many_to_one', indicator=True
)

display(df['_merge'].value_counts().to_frame('rows'))
print(f"Matched: {(df['_merge'].eq('both')).mean():.1%}")
df = df.drop(columns='_merge')

## 5. ניקוי ובניית משתני סגנון

נתונים למשחק מושפעים מזמן המשחק. לכן ניצור נתונים **ל־36 דקות**, המתאימים יותר להשוואת סגנון בין שחקנים. אחוזי קליעה נשמרים כפי שהם. לא נשתמש בנקודות גולמיות, במדד valuation או ב־plus/minus כמשתני אשכול מרכזיים, משום שהם מערבבים תפקיד עם איכות, דקות והצלחת הקבוצה.

נשמור רק שחקן–עונה ששיחק לפחות 300 דקות ו־10 משחקים. אפשר לבצע בדיקת רגישות לסף זה בהמשך.

In [ ]:
count_cols = {
    'points':'pts36', 'two_points_attempted':'two_pa36',
    'three_points_attempted':'three_pa36', 'free_throws_attempted':'fta36',
    'offensive_rebounds':'orb36', 'defensive_rebounds':'drb36',
    'assists':'ast36', 'steals':'stl36', 'turnovers':'tov36',
    'blocks_favour':'blk36', 'blocks_against':'blk_against36',
    'fouls_committed':'pf36', 'fouls_received':'fd36'
}

for raw, new in count_cols.items():
    df[new] = np.where(df['minutes'] > 0, df[raw] / df['minutes'] * 36, np.nan)

df['three_point_attempt_share'] = df['three_points_attempted'] / (
    df['two_points_attempted'] + df['three_points_attempted']).replace(0, np.nan)
df['assist_turnover_ratio'] = df['assists'] / df['turnovers'].replace(0, np.nan)
df['start_rate'] = df['games_started'] / df['games_played'].replace(0, np.nan)

features = ['pts36','two_pa36','three_pa36','fta36','orb36','drb36',
            'ast36','stl36','tov36','blk36','pf36','fd36',
            'three_point_attempt_share']

model_df = df[(df['minutes'] >= 300) & (df['games_played'] >= 10)].copy()
model_df = model_df.dropna(subset=features)
print('Rows before filtering:', len(df))
print('Rows used for clustering:', len(model_df))
print('Unique players:', model_df['player_id'].nunique())
print('Seasons:', model_df['season_code'].min(), 'to', model_df['season_code'].max())

## 6. חקירה ראשונית

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
sns.histplot(model_df['height_cm'].dropna(), bins=20, ax=axes[0])
axes[0].set_title('Height distribution')
sns.countplot(data=model_df, x='position', order=model_df['position'].value_counts().index, ax=axes[1])
axes[1].set_title('Traditional positions')
model_df.groupby('season_code').size().plot(ax=axes[2], marker='o')
axes[2].set_title('Observations by season')
axes[2].tick_params(axis='x', rotation=90)
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(11, 8))
sns.heatmap(model_df[features].corr(), cmap='vlag', center=0)
plt.title('Feature correlations')
plt.tight_layout(); plt.show()

## 7. תקנון ו־PCA

K-Means מבוסס על מרחקים, ולכן חובה לתקנן את המשתנים. PCA אינו משמש כאן ליצירת התוויות המסורתיות אלא להצגה דו־ממדית ולבדיקת המבנה הכללי.

In [ ]:
X = model_df[features].copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca_full = PCA().fit(X_scaled)
cum_var = np.cumsum(pca_full.explained_variance_ratio_)
plt.figure(figsize=(8,4))
plt.plot(range(1, len(cum_var)+1), cum_var, marker='o')
plt.axhline(.80, color='red', linestyle='--', label='80%')
plt.xlabel('Number of components'); plt.ylabel('Cumulative explained variance')
plt.legend(); plt.show()

pca2 = PCA(n_components=2)
coords = pca2.fit_transform(X_scaled)
model_df[['PC1','PC2']] = coords
print('Variance explained by 2 PCs:', pca2.explained_variance_ratio_.sum().round(3))

## 8. בחירת מספר האשכולות

לא קיים מדד יחיד שמוכיח מהו מספר האשכולות "האמיתי". נשווה בין 2–10 אשכולות באמצעות:
- Silhouette: גבוה יותר עדיף.
- Calinski–Harabasz: גבוה יותר עדיף.
- Davies–Bouldin: נמוך יותר עדיף.
- Inertia: משמשת לזיהוי "מרפק".

הבחירה הסופית צריכה לשלב מדדים, יציבות ויכולת לתת לאשכולות פרשנות מקצועית.

In [ ]:
scores = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=30)
    labels = km.fit_predict(X_scaled)
    scores.append({
        'k': k, 'silhouette': silhouette_score(X_scaled, labels),
        'calinski_harabasz': calinski_harabasz_score(X_scaled, labels),
        'davies_bouldin': davies_bouldin_score(X_scaled, labels),
        'inertia': km.inertia_
    })
scores = pd.DataFrame(scores)
display(scores.round(3))

fig, axes = plt.subplots(2,2,figsize=(13,8))
for ax, col, title in zip(axes.ravel(),
    ['silhouette','calinski_harabasz','davies_bouldin','inertia'],
    ['Silhouette ↑','Calinski-Harabasz ↑','Davies-Bouldin ↓','Inertia (elbow)']):
    sns.lineplot(data=scores, x='k', y=col, marker='o', ax=ax)
    ax.set_title(title)
plt.tight_layout(); plt.show()

BEST_K = int(scores.loc[scores['silhouette'].idxmax(), 'k'])
print('Suggested k by Silhouette:', BEST_K)

### בחירה ידנית אפשרית
אם הפתרון האוטומטי יוצר מעט מדי אשכולות רחבים, ניתן לשנות את `CHOSEN_K` לאחר עיון במדדים. אין לבחור מספר רק משום שהוא דומה לחמש העמדות המסורתיות.

In [ ]:
CHOSEN_K = BEST_K  # אפשר להחליף במספר בין 2 ל-10 לאחר עיון בתוצאות
kmeans = KMeans(n_clusters=CHOSEN_K, random_state=RANDOM_STATE, n_init=50)
model_df['cluster'] = kmeans.fit_predict(X_scaled)
print(model_df['cluster'].value_counts().sort_index())

## 9. הצגת האשכולות

In [ ]:
plt.figure(figsize=(10,7))
sns.scatterplot(data=model_df, x='PC1', y='PC2', hue='cluster', palette='tab10', alpha=.65)
plt.title(f'K-Means clusters (k={CHOSEN_K}) on PCA projection')
plt.legend(title='Cluster', bbox_to_anchor=(1.02,1)); plt.tight_layout(); plt.show()

In [ ]:
cluster_profiles_z = pd.DataFrame(
    kmeans.cluster_centers_, columns=features,
    index=[f'Cluster {i}' for i in range(CHOSEN_K)]
)

plt.figure(figsize=(14, max(4, CHOSEN_K*.7)))
sns.heatmap(cluster_profiles_z, cmap='vlag', center=0, annot=True, fmt='.1f')
plt.title('Cluster profiles in standard deviations from the sample mean')
plt.xlabel('Playing-style feature'); plt.ylabel('Cluster')
plt.tight_layout(); plt.show()

cluster_profiles_raw = model_df.groupby('cluster')[features + ['height_cm','minutes_per_game']].mean()
display(cluster_profiles_raw.round(2))

## 10. פרשנות מקצועית

הקוד הבא מציג בכל אשכול את המאפיינים החריגים ביותר ואת השחקן–עונות הקרובים ביותר למרכז. על בסיסם יש לתת שמות מקצועיים כגון "רכז יוצר", "גבוה מרווח", "מסיים בצבע" או "כנף דו־כיוונית". השמות הם תוצאה פרשנית של הנתונים, ולא תוויות שניתנו מראש.

In [ ]:
for c in range(CHOSEN_K):
    center = kmeans.cluster_centers_[c]
    high = cluster_profiles_z.loc[f'Cluster {c}'].nlargest(4)
    low = cluster_profiles_z.loc[f'Cluster {c}'].nsmallest(3)
    idx = np.where(model_df['cluster'].to_numpy() == c)[0]
    dist = np.linalg.norm(X_scaled[idx] - center, axis=1)
    examples = model_df.iloc[idx[np.argsort(dist)[:8]]][
        ['player','season_code','team_id','position','height_cm']]
    print(f'\n=== CLUSTER {c} ===')
    print('High:', ', '.join(f'{k} ({v:+.2f})' for k,v in high.items()))
    print('Low: ', ', '.join(f'{k} ({v:+.2f})' for k,v in low.items()))
    display(examples)

In [ ]:
# מלאו לאחר בחינת הפרופילים והדוגמאות
cluster_names = {c: f'Role {c}' for c in range(CHOSEN_K)}
# דוגמה בלבד: cluster_names = {0:'רכז יוצר', 1:'גבוה מסיים', ...}
model_df['role_name'] = model_df['cluster'].map(cluster_names)
display(model_df[['cluster','role_name']].drop_duplicates().sort_values('cluster'))

## 11. השוואה לעמדות המסורתיות

העמדה המסורתית משמשת כאן כמשתנה חיצוני. אם בכל אשכול מופיעות כמה עמדות מסורתיות, או אם עמדה אחת מתפזרת בין כמה אשכולות, זו עדות לכך שהחלוקה החדשה תופסת הבדלים שהחלוקה הקלאסית אינה מתארת.

In [ ]:
comparison = pd.crosstab(model_df['cluster'], model_df['position'], normalize='index')
display((comparison*100).round(1).astype(str) + '%')

plt.figure(figsize=(9, max(4, CHOSEN_K*.6)))
sns.heatmap(comparison, annot=True, fmt='.0%', cmap='Blues')
plt.title('Traditional position composition within each discovered cluster')
plt.ylabel('Cluster'); plt.tight_layout(); plt.show()

valid = model_df.dropna(subset=['position'])
traditional_codes = pd.factorize(valid['position'])[0]
print('ARI:', round(adjusted_rand_score(traditional_codes, valid['cluster']), 3))
print('NMI:', round(normalized_mutual_info_score(traditional_codes, valid['cluster']), 3))

## 12. שינוי תפקיד לאורך הקריירה

In [ ]:
multi = model_df.groupby('player_id').filter(lambda g: len(g) >= 3)
changes = (multi.sort_values(['player_id','season_code'])
           .groupby(['player_id','player'])['cluster']
           .agg(seasons='size', unique_roles='nunique', role_sequence=lambda x: ' → '.join(map(str,x)))
           .query('unique_roles > 1')
           .sort_values(['unique_roles','seasons'], ascending=False))
display(changes.head(25))

## 13. בדיקת מודל חלופי: Gaussian Mixture

GMM מאפשר השתייכות הסתברותית: שחקן ורסטילי עשוי להיות קרוב ליותר מתפקיד אחד. נשווה את הפתרון ל־K-Means ונציג את מידת הוודאות.

In [ ]:
gmm = GaussianMixture(n_components=CHOSEN_K, covariance_type='full',
                      random_state=RANDOM_STATE, n_init=10)
gmm_labels = gmm.fit_predict(X_scaled)
gmm_prob = gmm.predict_proba(X_scaled)
model_df['gmm_cluster'] = gmm_labels
model_df['gmm_confidence'] = gmm_prob.max(axis=1)

print('Agreement KMeans–GMM (ARI):', round(adjusted_rand_score(model_df['cluster'], gmm_labels),3))
display(model_df.nsmallest(15, 'gmm_confidence')[
    ['player','season_code','position','cluster','gmm_cluster','gmm_confidence']])

## 14. צירוף עתידי של נתוני 365Scores

בקובץ 365Scores כנראה לא יהיה `player_id` תואם. לכן נבצע התאמה בשלבים:
1. נרמול שמות והסרת סימני פיסוק/מבטאים.
2. התאמה מדויקת של השם המנורמל.
3. רק לרשומות שלא הותאמו: התאמה מטושטשת והפקת מועמדים.
4. שימוש בקבוצה, עונה ומידע נוסף לאימות.
5. אישור ידני של התאמות גבוליות לפני החיבור.

**אין לחבר אוטומטית רק לפי הציון הגבוה ביותר**, משום ששמות חלקיים עלולים להתאים ליותר משחקן אחד.

In [ ]:
from rapidfuzz import process, fuzz

def normalize_name(value):
    if pd.isna(value): return ''
    s = unicodedata.normalize('NFKD', str(value))
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    s = re.sub(r'[^a-zA-Z0-9 ]+', ' ', s.lower())
    return ' '.join(s.split())

# לאחר יצירת הקובץ, העלו אותו והחליפו את שם העמודה לפי הצורך:
# scores365 = pd.read_csv('players_365scores.csv')
# scores365['name_norm'] = scores365['player_name'].map(normalize_name)
# reference = bio_one.assign(full_name=bio_one['first_name'].fillna('')+' '+bio_one['last_name'].fillna(''))
# reference['name_norm'] = reference['full_name'].map(normalize_name)
# exact = scores365.merge(reference[['player_id','name_norm']], on='name_norm', how='left')

In [ ]:
def fuzzy_candidates(source_names, reference_df, limit=3, min_score=70):
    choices = reference_df['name_norm'].dropna().unique().tolist()
    rows = []
    for original in source_names:
        norm = normalize_name(original)
        for candidate, score, _ in process.extract(norm, choices, scorer=fuzz.token_set_ratio, limit=limit):
            if score >= min_score:
                ids = reference_df.loc[reference_df['name_norm'].eq(candidate), 'player_id'].tolist()
                rows.append({'name_365':original, 'name_norm':norm,
                             'candidate_norm':candidate, 'score':score,
                             'candidate_player_ids':ids})
    return pd.DataFrame(rows)

# review_table = fuzzy_candidates(
#     exact.loc[exact['player_id'].isna(), 'player_name'].dropna().unique(), reference)
# display(review_table.sort_values(['name_365','score'], ascending=[True,False]))
# review_table.to_csv('365scores_matches_for_manual_review.csv', index=False)

## 15. ייצוא תוצאות

In [ ]:
output_cols = ['season_player_id','season_code','player_id','player','team_id',
               'height_cm','position','cluster','role_name','PC1','PC2'] + features
model_df[output_cols].to_csv('euroleague_discovered_roles.csv', index=False)
cluster_profiles_raw.to_csv('cluster_profiles.csv')
scores.to_csv('cluster_validation_scores.csv', index=False)
print('Exported analysis files.')

try:
    from google.colab import files
    files.download('euroleague_discovered_roles.csv')
except Exception:
    pass

## 16. מסקנות שיש להשלים לאחר ההרצה

- מספר האשכולות שנבחר והנימוק לבחירה.
- המאפיינים המבדילים בין התפקידים שהתגלו.
- דוגמאות לשחקנים מייצגים בכל תפקיד.
- מידת החפיפה בין התפקידים החדשים לעמדות המסורתיות.
- שחקנים ורסטיליים או גבוליים לפי הסתברויות GMM.
- שינויי תפקיד אצל אותו שחקן בין עונות.

### מגבלות
- הנתונים מתארים פעולות שהסתיימו בסטטיסטיקה ואינם כוללים מיקום על המגרש, חסימות ללא כדור או משימות הגנתיות מלאות.
- נתוני per-36 מצמצמים את השפעת דקות המשחק אך אינם מבטלים הבדלים באיכות היריבה, קצב המשחק ומבנה הקבוצה.
- אותה תצפית אינה שחקן ייחודי אלא שחקן–עונה–קבוצה; יש להתחשב בתלות בין עונות של אותו שחקן.
- בחירת מספר האשכולות היא החלטה אנליטית ופרשנית, לא אמת מוחלטת.
- עמדות מאתר חיצוני עשויות להשתנות בין מקורות ובין עונות.